# Step 2: Compute final metrics from Step 1 outputs

Reads `refusal.json`, `vu.json`, `acc.json` produced by `eval_second_step.ipynb` (matched by `name` = question).  
Optionally reads SU from a CSV for VU/SU disagreement and correlation.  
Writes `metrics.json` into the same folder.

## 0. Config

In [ ]:
!pip install -q python-dotenv

In [ ]:
import os, json
import numpy as np
import pandas as pd
from scipy import stats
from dotenv import load_dotenv

load_dotenv(dotenv_path=os.path.join(os.getcwd(), "..", ".env"))

# ── User config ──────────────────────────────────────────────────────────────
STEP1_DIR  = "sae_steering_output/"
SU_CSV     = "SU_Amir.csv"
SU_COLUMN  = "sentence_semantic_entropy"
# ─────────────────────────────────────────────────────────────────────────────

METRICS_OUT = STEP1_DIR + "metrics.json"

## 1. Load Step 1 outputs

In [2]:
def load(path):
    with open(path, encoding="utf-8") as f:
        return json.load(f)

def normalize_q(q: str) -> str:
    """Strip whitespace and trailing '?' for robust cross-file matching."""
    return q.strip().rstrip("?").strip()

refusal_data = load(STEP1_DIR + "refusal.json")   # [{"name": q, "refusal": bool}, ...]
vu_data      = load(STEP1_DIR + "vu.json")         # [{"name": q, "vu": float}, ...]
acc_path     = STEP1_DIR + "acc.json"
acc_data     = load(acc_path) if os.path.exists(acc_path) else None

# Build normalized_name -> value lookups
q2refusal = {normalize_q(r["name"]): r["refusal"] for r in refusal_data}
q2vu      = {normalize_q(r["name"]): r["vu"]      for r in vu_data}
q2correct = {normalize_q(r["name"]): r["correct"] for r in acc_data} if acc_data else {}

# SU from CSV
q2su = {}
if SU_CSV and os.path.exists(SU_CSV):
    df = pd.read_csv(SU_CSV)
    for _, row in df.iterrows():
        q2su[normalize_q(str(row["question"]))] = float(row[SU_COLUMN])

# Valid questions: those with parseable VU
questions = [normalize_q(r["name"]) for r in vu_data if r["vu"] >= 0]
print(f"Samples with valid VU: {len(questions)}")
print(f"acc available:         {bool(q2correct)}")
print(f"SU available:          {bool(q2su)}")
su_matched = sum(1 for q in questions if q in q2su)
print(f"SU matched:            {su_matched}/{len(questions)}")

Samples with valid VU: 500
acc available:         True
SU available:          True
SU matched:            500/500


## 2. Compute metrics

In [3]:
def kmeans1d_threshold(values):
    """1D k-means boundary (minimises within-cluster sum of squares)."""
    vals = np.array(values)
    c0, c1 = np.percentile(vals, 25), np.percentile(vals, 75)
    for _ in range(200):
        labels = (np.abs(vals - c1) < np.abs(vals - c0)).astype(int)
        new_c0 = vals[labels == 0].mean() if (labels == 0).any() else c0
        new_c1 = vals[labels == 1].mean() if (labels == 1).any() else c1
        if np.abs(new_c0 - c0) < 1e-9 and np.abs(new_c1 - c1) < 1e-9:
            break
        c0, c1 = new_c0, new_c1
    return (c0 + c1) / 2.0

n = len(questions)

vu_arr      = np.array([q2vu[q]      for q in questions])
refusal_arr = np.array([q2refusal.get(q, False) for q in questions])
correct_arr = np.array([q2correct.get(q, None)  for q in questions])  # None if no acc
su_arr      = np.array([q2su.get(q, float("nan")) for q in questions])

vu_threshold = kmeans1d_threshold(vu_arr)
su_vals_valid = su_arr[~np.isnan(su_arr)]
su_threshold  = kmeans1d_threshold(su_vals_valid) if len(su_vals_valid) > 1 else None

print(f"VU threshold: {vu_threshold:.4f}")
print(f"SU threshold: {su_threshold:.4f}" if su_threshold else "SU threshold: n/a")

# Refusal Rate
refusal_rate = float(refusal_arr.mean())

# Correctness / hallucination (requires acc)
has_acc = (correct_arr[0] is not None)
if has_acc:
    correct_arr = correct_arr.astype(bool)
    correctness_rate   = float(correct_arr.mean())
    overall_hall       = float((~refusal_arr & ~correct_arr).mean())
    confident_hall     = float((~correct_arr & (vu_arr < vu_threshold)).mean())
    vu_for_correct     = float(vu_arr[correct_arr   & ~refusal_arr].mean()) if (correct_arr & ~refusal_arr).any()   else float("nan")
    vu_for_incorrect   = float(vu_arr[~correct_arr  & ~refusal_arr].mean()) if (~correct_arr & ~refusal_arr).any()  else float("nan")
else:
    correctness_rate = overall_hall = confident_hall = float("nan")
    vu_for_correct = vu_for_incorrect = float("nan")

# VU/SU disagreement & correlation
if su_threshold is not None:
    mask = ~np.isnan(su_arr)
    vu_p, su_p = vu_arr[mask], su_arr[mask]
    disagreement_rate = float(np.mean((vu_p >= vu_threshold) != (su_p >= su_threshold)))
    corr = float(stats.pearsonr(vu_p, su_p)[0])
else:
    disagreement_rate = corr = float("nan")

metrics = {
    "n_samples":                     n,
    "vu_threshold":                  round(vu_threshold, 6),
    "su_threshold":                  round(su_threshold, 6) if su_threshold else None,
    "refusal_rate":                  round(refusal_rate, 6),
    "correctness_rate":              round(correctness_rate, 6),
    "overall_hallucination_rate":    round(overall_hall, 6),
    "confident_hallucination_rate":  round(confident_hall, 6),
    "vu_su_disagreement_rate":       round(disagreement_rate, 6),
    "vu_su_correlation":             round(corr, 6),
    "vu_for_correct_answer":         round(vu_for_correct, 6),
    "vu_for_incorrect_answer":       round(vu_for_incorrect, 6),
}

with open(METRICS_OUT, "w", encoding="utf-8") as f:
    json.dump(metrics, f, ensure_ascii=False, indent=2)
print(f"Saved → {METRICS_OUT}\n")
print("=" * 55)
for k, v in metrics.items():
    print(f"  {k:<38} {v}")
print("=" * 55)

VU threshold: 0.4665
SU threshold: 1.1346
Saved → pre_steering_output/metrics.json

  n_samples                              500
  vu_threshold                           0.466485
  su_threshold                           1.134634
  refusal_rate                           0.044
  correctness_rate                       0.506
  overall_hallucination_rate             0.45
  confident_hallucination_rate           0.38
  vu_su_disagreement_rate                0.404
  vu_su_correlation                      0.319071
  vu_for_correct_answer                  0.045455
  vu_for_incorrect_answer                0.186889
